<a href="https://colab.research.google.com/github/411271108/1/blob/main/%E5%88%A4%E5%88%A5%E7%9F%B3%E8%99%8E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os

# ==========================================
# 步驟 1: 搭建核心神經網絡 (遷移學習)
# ==========================================
def build_leopard_cat_model():
    # 我們使用 MobileNetV2 作為基底，它是一個輕量級但強大的影像辨識模型
    # include_top=False 表示我們不要它原本的最後分類層
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                                   include_top=False,
                                                   weights='imagenet')

    # 凍結基底模型的權重，防止它們在初次訓練中被破壞
    base_model.trainable = False

    # 搭建我們自己的分類頭 (這就是專門判斷石虎的「神經元」部分)
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(), # 將特徵圖轉換為單一向量
        layers.Dense(128, activation='relu'), # 一個中間隱藏層，增強表達能力
        layers.Dropout(0.2), # 防止過度擬合
        # 最後一個神經元，使用 Sigmoid 激活函數，輸出 0~1 之間的機率值
        # 0 代表「家貓」，1 代表「石虎」
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

# ==========================================
# 步驟 2: 載入模型權重
# ==========================================
# 在實際應用中，你需要先用石虎和家貓的圖片訓練這個模型。
# 這裡我們模擬一個已經訓練好的模型。
# 為了讓你現在能運行，程式會生成一個「未經石虎數據專門訓練」的模型。
# (注意：沒有經過專門數據訓練的模型，辨識石虎的準確率會很低)

print("正在初始化石虎辨識模型...")
my_model = build_leopard_cat_model()
# 如果你已經訓練好並保存了模型，取消下方的註釋
# my_model.load_weights('leopard_cat_weights.h5')
print("模型初始化完成。")

# ==========================================
# 步驟 3: 圖像處理與推理函數
# ==========================================
def predict_if_leopard_cat(image_path):
    """
    輸入圖片路徑，判定是否為石虎。
    """
    if not os.path.exists(image_path):
        print(f"錯誤：找不到圖片檔案 {image_path}")
        return

    # 1. 載入圖片並調整大小以符合模型輸入 (224x224)
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((224, 224))

    # 2. 轉為 Numpy 陣列並進行預處理
    img_array = np.array(img_resized)
    # MobileNetV2 的預處理：將像素值歸一化到 [-1, 1] 之間
    img_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)

    # 3. 增加一個維度 (Batch dimension)，變成 (1, 224, 224, 3)
    img_batch = np.expand_dims(img_preprocessed, axis=0)

    # 4. 進行推理 (Prediction)
    print(f"正在分析圖片：{image_path}...")
    prediction = my_model.predict(img_batch)
    score = prediction[0][0] # 取得 Sigmoid 的輸出值

    # 5. 顯示結果
    plt.imshow(img)
    plt.axis('off')

    # 這裡設置一個門檻值 (Threshold)，通常是 0.5
    if score > 0.5:
        result = f"判定結果：【石虎】 (信心度: {score*100:.1f}%)"
        color = 'green'
    else:
        result = f"判定結果：【一般貓咪】 (信心度: {(1-score)*100:.1f}%)"
        color = 'red'

    plt.title(result, color=color, fontsize=12)
    plt.show()

    print("-" * 30)
    print(f"原始神經元輸出值 (0~1): {score:.4f}")
    print(result)
    print("-" * 30)

# ==========================================
# 步驟 4: 實際測試 (請將你自己的圖片上傳並修改路徑)
# ==========================================

# 模擬：你現在丢一張圖片給它。
# 1. 打開你的圖片資料夾
# 2. 找到你想測試的圖片 (例如: 'my_cat.jpg' 或 'wildlife_camera_photo.png')
# 3. 將圖片放到與此程式碼相同的資料夾中，或者提供完整路徑。

# 範例調用 (你需要將下方路徑替換為你實際的圖片路徑)
# predict_if_leopard_cat('test_image_1.jpg')

正在初始化石虎辨識模型...
模型初始化完成。


In [ ]:
# 刪除 dataset 裡面所有隱藏的 checkpoints 資料夾
!rm -rf `find dataset -name .ipynb_checkpoints`

In [ ]:
import os
import zipfile
import requests

# ==========================================
# 1. 下載範例資料集 (這裡模擬下載一個小型石虎/貓咪壓縮檔)
# ==========================================
# 註：為了示範，我們建立資料夾並模擬一些資料流
if not os.path.exists('dataset'):
    os.makedirs('dataset/leopard_cat', exist_ok=True)
    os.makedirs('dataset/domestic_cat', exist_ok=True)
    print("資料夾已建立！請將照片上傳至對應資料夾。")

# ==========================================
# 2. 訓練設定 (Data Augmentation)
# ==========================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 這段會把照片旋轉、縮放，讓神經元學得更紮實
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# 讀取剛剛建立的資料夾
try:
    train_data = train_gen.flow_from_directory(
        'dataset',
        target_size=(224, 224),
        batch_size=4,
        class_mode='binary',
        subset='training'
    )

    # ==========================================
    # 3. 執行特訓 (Fit)
    # ==========================================
    print("\n--- 神經元特訓開始 ---")
    # epochs=10 代表看 10 遍。如果照片少，這很快就結束了
    my_model.fit(train_data, epochs=30)
    print("--- 特訓結束，神經元已獲得經驗值 ---")

except Exception as e:
    print(f"訓練失敗：{e}")
    print("提示：請至少在 dataset/cat 和 dataset/leopard_cat 裡各放 2-3 張照片喔！")

Found 0 images belonging to 2 classes.

--- 神經元特訓開始 ---
訓練失敗：The PyDataset has length 0
提示：請至少在 dataset/cat 和 dataset/leopard_cat 裡各放 2-3 張照片喔！


In [ ]:
# 檢查神經元學到了多少東西
print(f"訓練集樣本數: {train_data.samples}")
print(f"標籤對應: {train_data.class_indices}")
# 應該顯示 {'cat': 0, 'leopard_cat': 1}，這代表 0 是貓，1 是石虎

訓練集樣本數: 67
標籤對應: {'domestic_cat': 0, 'leopard_cat': 1}


In [16]:
from google.colab import files

# 1. 彈出視窗讓你選圖片
uploaded = files.upload()

# 2. 自動抓取你剛剛選的檔案名稱，並「呼叫」判定函數
for filename in uploaded.keys():
    predict_if_leopard_cat(filename)

KeyboardInterrupt: 